In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")

print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
%pip install -q \
    transformers==4.51.3 \
    peft==0.15.2 \
    accelerate==1.6.0 \
    huggingface-hub==0.36.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
import torch
import transformers
import peft
import accelerate

assert transformers.__version__ == "4.51.3"
assert peft.__version__ == "0.15.2"
assert accelerate.__version__ == "1.6.0"
assert torch.cuda.is_available()

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("PASS: comparison environment ready.")

Transformers: 4.51.3
PEFT: 0.15.2
Accelerate: 1.6.0
GPU: Tesla T4
PASS: comparison environment ready.


In [6]:
from google.colab import files
from pathlib import Path
import json
import zipfile

print("First, select the experiment ZIP.")
uploaded = files.upload()

print("Next, select the local development JSON.")
uploaded.update(files.upload())

# Dynamically find the keys to handle duplicate suffixes like " (2)" appended by Colab
zip_name = next((k for k in uploaded.keys() if k.startswith("fincite-experiment-20261002T213157Z") and k.endswith(".zip")), "fincite-experiment-20261002T213157Z.zip")
run_name = next((k for k in uploaded.keys() if k.startswith("local-development-2026-10-07T18-59-53-043Z") and k.endswith(".json")), "local-development-2026-10-07T18-59-53-043Z.json")

assert zip_name in uploaded, "Upload the experiment ZIP."
assert run_name in uploaded, "Upload the local development JSON."

restore_dir = Path("/content/fincite-controlled-comparison")
restore_dir.mkdir(parents=True, exist_ok=True)

# Restore only the selected adapter and training report.
with zipfile.ZipFile(zip_name) as archive:
    selected = [
        name for name in archive.namelist()
        if not name.endswith("/")
        and (
            "/adapter-best/" in name
            or name.endswith("/training-report.json")
        )
    ]
    assert selected, "Selected adapter was not found in the ZIP."

    for name in selected:
        target = (restore_dir / name).resolve()
        assert target.is_relative_to(restore_dir.resolve()), "Invalid ZIP path."
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))

adapters = list(restore_dir.rglob("adapter-best/adapter_config.json"))
assert len(adapters) == 1, "Expected one selected adapter."
adapter_dir = adapters[0].parent

assert (adapter_dir / "adapter_model.safetensors").is_file()

reports = list(restore_dir.rglob("training-report.json"))
assert len(reports) == 1
training_report = json.loads(reports[0].read_text(encoding="utf-8"))
assert training_report["selected_epoch"] == 3

local_run = json.loads(Path(run_name).read_text(encoding="utf-8-sig"))
assert len(local_run["results"]) == 13
assert all(
    row["http_status"] == 200 and row["api_response"]["sources"]
    for row in local_run["results"]
)

print("PASS: selected epoch 3 adapter restored.")
print("PASS: 13 development responses and supplied passages loaded.")
print("Adapter folder:", adapter_dir)

First, select the experiment ZIP.


Saving fincite-experiment-20261002T213157Z.zip to fincite-experiment-20261002T213157Z (3).zip
Next, select the local development JSON.


Saving local-development-2026-10-07T18-59-53-043Z.json to local-development-2026-10-07T18-59-53-043Z (3).json
PASS: selected epoch 3 adapter restored.
PASS: 13 development responses and supplied passages loaded.
Adapter folder: /content/fincite-controlled-comparison/fincite-experiment-20261002T213157Z/adapter-best


In [7]:
import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from peft import PeftModel, get_peft_model_state_dict
from safetensors.torch import load_file
from google.colab import files

MODEL_ID = "Qwen/Qwen3-1.7B"
REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"

# Use the latest uploaded bytes, even if Colab renamed the file.
run_bytes = bytes(uploaded[run_name])
local_run = json.loads(run_bytes.decode("utf-8-sig"))
assert len(local_run["results"]) == 13
assert torch.cuda.is_available()

system_prompt = (
    "Answer using only the supplied passages. Keep the answer brief and "
    "preserve conditions and exceptions. Cite supported factual claims "
    "using [source_id]. If the passages cannot answer the question, "
    "explain that limitation without guessing or inventing a citation."
)

print("Loading the pinned base model and saved adapter...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda")

model = PeftModel.from_pretrained(
    base_model, str(adapter_dir), is_trainable=False
)
model.eval()

# Verify the loaded adapter against the saved weights.
adapter_path = adapter_dir / "adapter_model.safetensors"
saved_weights = load_file(str(adapter_path), device="cpu")
loaded_weights = get_peft_model_state_dict(model)

assert set(saved_weights) == set(loaded_weights)
for key, saved in saved_weights.items():
    loaded = loaded_weights[key].detach().cpu()
    assert torch.isfinite(loaded).all(), f"Non-finite weight: {key}"
    assert torch.equal(saved, loaded.to(saved.dtype)), f"Weight mismatch: {key}"

print("PASS: adapter weights match the saved checkpoint.")

generation_config = GenerationConfig(
    do_sample=False,
    num_beams=1,
    max_new_tokens=256,
    temperature=1.0,
    top_p=1.0,
    top_k=50,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

def generate(inputs):
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            generation_config=generation_config,
        )
    torch.cuda.synchronize()

    generated = output[0, inputs["input_ids"].shape[1]:]
    return {
        "answer": tokenizer.decode(
            generated, skip_special_tokens=True
        ).strip(),
        "generated_tokens": len(generated),
        "reached_token_limit": (
            len(generated) >= generation_config.max_new_tokens
            and generated[-1].item() != tokenizer.eos_token_id
        ),
        "generation_seconds": round(time.perf_counter() - started, 3),
    }

report = {
    "status": "in_progress",
    "purpose": "controlled_development_comparison",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_revision": REVISION,
    "adapter_sha256": hashlib.sha256(adapter_path.read_bytes()).hexdigest(),
    "source_run_sha256": hashlib.sha256(run_bytes).hexdigest(),
    "precision": "float16 base; adapter as loaded by PEFT",
    "attention": "sdpa",
    "enable_thinking": False,
    "generation_settings": generation_config.to_dict(),
    "evidence_note": (
        "Messages reconstructed from saved API source quotes. "
        "Source grouping may differ from the original local request. "
        "Base and adapter receive identical messages and tokens."
    ),
    "grading_status": "ungraded",
    "results": [],
}

output_path = Path("/content/fincite-controlled-comparison-v01.json")

def save_report():
    output_path.write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )

save_report()

for index, row in enumerate(local_run["results"], start=1):
    sources = row["api_response"]["sources"]
    passages = "\n\n".join(
        f'[{source["id"]}]\n{source["quote"]}'
        for source in sources
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                f'Question: {row["question"]}\n\n'
                f"Supplied passages:\n{passages}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=False,
    ).to("cuda")

    input_tokens = inputs["input_ids"].shape[1]
    assert input_tokens + 256 <= 2048, (
        f'{row["question_id"]}: context exceeds the comparison limit.'
    )

    with model.disable_adapter():
        base_answer = generate(inputs)

    adapter_answer = generate(inputs)

    report["results"].append({
        "question_id": row["question_id"],
        "question": row["question"],
        "answerability": row["answerability"],
        "reference_claims": row["reference_claims"],
        "forbidden_claims": row["forbidden_claims"],
        "sources": sources,
        "messages": messages,
        "input_tokens": input_tokens,
        "base": base_answer,
        "adapter": adapter_answer,
        "identical_answers": (
            base_answer["answer"] == adapter_answer["answer"]
        ),
    })
    save_report()
    print(f'{index}/13: {row["question_id"]} saved.')

report["status"] = "completed_ungraded"
report["finished_at"] = datetime.now(timezone.utc).isoformat()
save_report()

print("\nDONE: all 13 base-versus-adapter comparisons saved.")
files.download(str(output_path))

Loading the pinned base model and saved adapter...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True, 'temperature': 0.6, 'top_k': 20, 'top_p': 0.95, 'bos_token_id': 151643}. If this is not desired, please set these values explicitly.


PASS: adapter weights match the saved checkpoint.
1/13: dev-001 saved.
2/13: dev-002 saved.
3/13: dev-003 saved.
4/13: dev-004 saved.
5/13: dev-005 saved.
6/13: dev-006 saved.
7/13: dev-007 saved.
8/13: dev-008 saved.
9/13: dev-009 saved.
10/13: dev-010 saved.
11/13: dev-011 saved.
12/13: dev-012 saved.
13/13: dev-013 saved.

DONE: all 13 base-versus-adapter comparisons saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>